# 6 · Databricks Platform & Unity Catalog  ·  *+ shared dataset setup*

Welcome to the **Databricks Data Engineering** core track. This first notebook
orients you in the platform and then **creates the shared "BrewBox" dataset that
every later notebook (7–15) reuses** — so you build it once here and never worry
about sample data again.

**Prerequisites:** basic SQL and Python. (Need those first? See the sibling
courses in `de-bootcamp-course`.)

**By the end you will:** know your way around Databricks notebooks, compute,
`dbutils`, and **Unity Catalog** (the governance layer), and have the BrewBox
schema, tables, and raw landing files ready to go.

> Run the cells top to bottom. Code cells are Python; they use the pre-injected
> `spark` session and `dbutils`, both provided automatically in a Databricks
> notebook.

## 1 · The Databricks notebook

A Databricks notebook is a sequence of **cells** attached to **compute**. Each
cell has a language; you switch it with a **magic command** on the first line:

| Magic | Cell runs as | Example use |
|---|---|---|
| `%python` | Python (the default) | PySpark, general code |
| `%sql` | SQL | `SELECT * FROM brewbox.customers` |
| `%md` | Markdown | notes & docs |
| `%sh` | Shell on the driver | `ls`, `pip` |
| `%fs` | File-system shortcut | `%fs ls /Volumes/...` |
| `%run` | Run another notebook inline | share setup/functions |

Two Databricks-only helpers you'll use constantly:

- **`display(df)`** — renders a rich, sortable, chartable table (use it instead of
  `df.show()` in Databricks).
- **`dbutils`** — utilities for the file system, secrets, widgets, and notebook
  workflow (more below).

Let's confirm where we are — `current_catalog()`, `current_schema()`, and who we
are:

In [ ]:
# `spark` is pre-injected on Databricks; this fallback lets the notebook load elsewhere.
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

spark.sql("SELECT current_catalog(), current_schema(), current_user() AS user").show(truncate=False)
print("Spark version:", spark.version)

## 2 · Compute — what actually runs your code

Your notebook does nothing until it's attached to compute. The main kinds:

- **Serverless compute** — instant, managed by Databricks; the simplest choice and
  the default on **Free Edition**.
- **All-purpose cluster** — a cluster you start for interactive work (notebooks).
- **Job cluster** — spun up for a scheduled job, then torn down (cheaper for
  pipelines).
- **SQL warehouse** — compute tuned for SQL/BI and `%sql` dashboards.

You (the **driver**) plan the work; **executors** run it in parallel across
partitions of the data (recap from the intro). For this course, **serverless** is
perfect — attach it from the top-right of the notebook.

## 3 · `dbutils` — the notebook toolbox

`dbutils` bundles small utilities. The ones a data engineer uses most:

```python
dbutils.fs.ls("/Volumes/…")          # list files/folders
dbutils.fs.mkdirs("/Volumes/…/x")    # make a directory
dbutils.fs.rm("path", recurse=True)  # delete
dbutils.widgets.text("date", "2024-01-01")   # notebook parameters (great for jobs)
dbutils.widgets.get("date")
dbutils.secrets.get("scope", "key")  # read a secret WITHOUT exposing it in code
dbutils.notebook.run("./other_nb", 60, {"date": "2024-01-01"})  # call another notebook
dbutils.notebook.exit("done")        # return a value to a caller
```

**Secrets** matter: never hard-code passwords/keys. Store them in a **secret
scope** and read them with `dbutils.secrets.get(...)` — Databricks redacts the
value in output. We'll use widgets to parameterize the capstone job later.

## 4 · Unity Catalog — governance & the 3-level namespace

**Unity Catalog (UC)** is Databricks' governance layer: one place to secure,
organize, and track *all* your data. Its key idea is a **three-level namespace**:

```
catalog . schema . table
   │         │        └── the table (or view)
   │         └── a group of tables (a.k.a. database)
   └── the top-level container (often per environment or team)
```

So a fully-qualified name looks like `main.brewbox.customers`. You set defaults
with `USE CATALOG` / `USE SCHEMA` so you can then write just `brewbox.customers`.

UC also governs:

- **Volumes** — governed storage for **files** (CSV, JSON, images, models) that
  live *outside* tables. Path form: `/Volumes/<catalog>/<schema>/<volume>/…`.
  (Volumes are the modern, governed replacement for the older **DBFS**.)
- **Permissions** — `GRANT SELECT ON TABLE … TO \`group\``; fine-grained access.
- **Lineage** — UC automatically tracks which tables/columns feed which, and which
  notebooks/jobs produced them.
- **`information_schema`** — query metadata about your objects with SQL.

Let's look at the catalogs and schemas available to you:

In [ ]:
spark.sql("SHOW CATALOGS").show(truncate=False)
# Everything below lands in the CURRENT catalog. Override it if needed:
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
print("Using catalog:", CATALOG, "-- set CATALOG = 'workspace' (or your catalog) if you hit permission errors.")

## 5 · Set up the shared **BrewBox** dataset  ☕  *(the important part)*

Now we create the dataset the whole track reuses: a fictional coffee chain,
**BrewBox**. We'll make:

- a **schema** `brewbox` to hold everything,
- a **Volume** `brewbox.landing` for **raw files**,
- **reference tables** — `customers`, `stores`, `products`, `order_items`,
- **raw landing files** — `orders/` and `events/` as JSON (these feed the
  ingestion notebook, `7`).

First, create the schema and the volume:

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS brewbox COMMENT 'Shared dataset for the Databricks DE track'")
spark.sql("CREATE VOLUME IF NOT EXISTS brewbox.landing COMMENT 'Raw landing files (orders, events)'")

LANDING = f"/Volumes/{CATALOG}/brewbox/landing"
print("Schema : brewbox")
print("Volume :", LANDING)

Now generate the data with plain Python (deterministic via a fixed seed), turn it
into Spark DataFrames, and persist it. Reference tables are written as **managed
Delta tables**; orders and events are written as **raw JSON files** into the
Volume.

In [ ]:
import random
from datetime import date, datetime, timedelta
random.seed(42)

COUNTRIES = ["US", "GB", "DE", "IN", "BR", "CA"]
REGIONS   = {"US": "North America", "CA": "North America", "GB": "Europe",
             "DE": "Europe", "IN": "Asia", "BR": "South America"}
CITIES    = ["Seattle", "London", "Berlin", "Mumbai", "Sao Paulo", "Toronto"]
CATS      = ["Hot Coffee", "Cold Coffee", "Bakery", "Merchandise"]
TIERS     = ["bronze", "silver", "gold"]
FIRST     = ["Ava","Liam","Mei","Noah","Olivia","Raj","Sofia","Chen","Amara","Ivan","Yara","Kofi"]
LAST      = ["Smith","Patel","Kim","Garcia","Rossi","Haddad","Silva","Wu","Okafor","Ivanov"]

# --- customers (40) ---
customers = []
for cid in range(1, 41):
    name = f"{random.choice(FIRST)} {random.choice(LAST)}"
    country = random.choice(COUNTRIES)
    email = None if random.random() < 0.1 else name.lower().replace(" ", ".") + "@example.com"
    signup = date(2023, 1, 1) + timedelta(days=random.randint(0, 700))
    customers.append((cid, name, country, email, signup, random.choice(TIERS)))
customers_df = spark.createDataFrame(
    customers, ["customer_id","name","country","email","signup_date","loyalty_tier"])

# --- stores (12) ---
stores = []
for sid in range(1, 13):
    c = random.randint(0, len(CITIES)-1)
    stores.append((f"S-{sid:02d}", f"BrewBox {CITIES[c]} #{sid}", CITIES[c], REGIONS[COUNTRIES[c]]))
stores_df = spark.createDataFrame(stores, ["store_id","store_name","city","region"])

# --- products (12) ---
names = ["Latte","Cappuccino","Cold Brew","Espresso","Mocha","Americano",
         "Croissant","Muffin","Cookie","Tumbler","Mug","Beans 1kg"]
products = []
for pid, nm in enumerate(names, start=101):
    cat = ("Bakery" if nm in ("Croissant","Muffin","Cookie")
           else "Merchandise" if nm in ("Tumbler","Mug","Beans 1kg")
           else "Cold Coffee" if "Cold" in nm else "Hot Coffee")
    products.append((pid, nm, cat, round(random.uniform(3.0, 25.0), 2)))
products_df = spark.createDataFrame(products, ["product_id","product_name","category","unit_price"])
price_of = {p[0]: p[3] for p in products}

# --- orders (300) + order_items ---
orders, items = [], []
start = datetime(2024, 1, 1)
for oid in range(1001, 1301):
    cid = random.randint(1, 40)
    sid = f"S-{random.randint(1,12):02d}"
    ts = start + timedelta(days=random.randint(0, 364), hours=random.randint(6, 21),
                           minutes=random.randint(0, 59))
    total = 0.0
    for _ in range(random.randint(1, 3)):
        pid = random.randint(101, 112); qty = random.randint(1, 4)
        total += qty * price_of[pid]
        items.append((oid, pid, qty, price_of[pid]))
    status = random.choices(["completed","returned","cancelled"], weights=[0.82,0.11,0.07])[0]
    orders.append((oid, cid, sid, ts.strftime("%Y-%m-%d %H:%M:%S"), status, round(total, 2)))
orders_df = spark.createDataFrame(
    orders, ["order_id","customer_id","store_id","order_ts","status","amount"])
items_df = spark.createDataFrame(items, ["order_id","product_id","quantity","unit_price"])

# --- events (500) ---
kinds = ["page_view","add_to_cart","checkout","search"]
ev_start = datetime(2024, 6, 1)
events = []
for eid in range(1, 501):
    ts = ev_start + timedelta(seconds=random.randint(0, 60*60*24*30))
    events.append((eid, random.randint(1, 40), random.choice(kinds), ts.strftime("%Y-%m-%d %H:%M:%S")))
events_df = spark.createDataFrame(events, ["event_id","customer_id","event_type","ts"])

print("Generated:",
      customers_df.count(), "customers,", stores_df.count(), "stores,",
      products_df.count(), "products,", orders_df.count(), "orders,",
      items_df.count(), "order_items,", events_df.count(), "events")

In [ ]:
# Persist REFERENCE tables as managed Delta tables in the brewbox schema
customers_df.write.mode("overwrite").saveAsTable("brewbox.customers")
stores_df.write.mode("overwrite").saveAsTable("brewbox.stores")
products_df.write.mode("overwrite").saveAsTable("brewbox.products")
items_df.write.mode("overwrite").saveAsTable("brewbox.order_items")

# Land RAW files in the Volume as JSON (multiple files -> nice for Auto Loader in nb 02)
orders_df.repartition(4).write.mode("overwrite").json(f"{LANDING}/orders")
events_df.repartition(2).write.mode("overwrite").json(f"{LANDING}/events")

print("Reference tables + raw landing files written.")

## 6 · Explore what you just created

List the tables in the schema, peek at a table, and list the raw files in the
Volume.

In [ ]:
spark.sql("SHOW TABLES IN brewbox").show(truncate=False)
display(spark.table("brewbox.customers").limit(5))   # display() = rich table on Databricks

In [ ]:
# The raw JSON files that notebook 02 will ingest with Auto Loader:
for f in dbutils.fs.ls(f"{LANDING}/orders"):
    print(f.name, f.size, "bytes")

In [ ]:
# You can query files directly with SQL too (schema is inferred):
spark.sql(f"SELECT * FROM json.`{LANDING}/orders` LIMIT 5").show()

## 7 · Exercises

Try each in the empty cell, then run the solution below it to check.

**Exercise 1 —** How many orders are in each `status`? (Query `brewbox` — but the
orders are still raw files, so read them from the Volume.)

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
spark.sql(f"""
    SELECT status, count(*) AS n
    FROM json.`{LANDING}/orders`
    GROUP BY status ORDER BY n DESC
""").show()

**Exercise 2 —** List the 5 most expensive products (name, category, unit_price)
from the `brewbox.products` table.

In [ ]:
# Your turn (Exercise 2):

In [ ]:
# ✅ Solution 2
from pyspark.sql.functions import desc
(spark.table("brewbox.products")
    .orderBy(desc("unit_price"))
    .select("product_name","category","unit_price")
    .show(5))

**Exercise 3 —** Use `DESCRIBE` to inspect the schema of `brewbox.customers`, and
count the customers with a missing email.

In [ ]:
# Your turn (Exercise 3):

In [ ]:
# ✅ Solution 3
spark.sql("DESCRIBE TABLE brewbox.customers").show(truncate=False)
spark.sql("SELECT count(*) AS missing_email FROM brewbox.customers WHERE email IS NULL").show()

## 8 · Recap & what's next

You now have:

- A working grasp of the Databricks **notebook**, **compute**, `dbutils`, and
  **Unity Catalog** (the `catalog.schema.table` namespace + **Volumes**).
- The shared **BrewBox** dataset in the `brewbox` schema: reference tables
  (`customers`, `stores`, `products`, `order_items`) and raw landing files
  (`orders/`, `events/`) in the `brewbox.landing` Volume.

Every following notebook builds on this. **Next → `7` Data Ingestion:** use
**Auto Loader** and `COPY INTO` to load those raw `orders` files into a **Bronze**
Delta table — the first layer of the Medallion pipeline. 🚀